# ARC-AGI-3 submission — dsh agent + Qwen3.8-27B-FP8 (并发版)

完整版: vLLM 本地起 Qwen3.8-27B(工具调用+关思考模板), deepseek-harness headless 通过
本地 game_server 打游戏; **N 局并发**(A3_CONCURRENCY, 默认 4): vLLM 批处理下各路速度几乎不掉,
每局摊到的开口次数 ≈ ×N。真提交走网关打隐藏游戏; Save & Run 用公开环境文件验证整条栈。


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen

NOTEBOOK_T0 = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
os.environ["ONLY_RESET_LEVELS"] = "true"
WORKING = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path("out")
WORKING.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("RECORDINGS_DIR", str(WORKING / "recordings"))

import torch
print("GPU:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))
assert torch.cuda.get_device_capability(0) >= (8, 9), "FP8 需要 CC>=8.9, 加速器要选 RTX 6000(NvidiaRtxPro6000)"
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


## 1. 离线装 vLLM + arc-agi


In [ ]:
def find_dir(pattern):
    for p in Path("/kaggle/input").rglob(pattern):
        return p.parent
    raise RuntimeError(f"找不到 {pattern}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links", str(find_dir("vllm-*.whl")), "vllm"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
                       "--no-warn-conflicts", "--find-links",
                       "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels", "arc-agi"])
import importlib.metadata as md_
print("vllm", md_.version("vllm"), "| arc-agi", md_.version("arc-agi"))


## 2. 定位源码 bundle 与模型权重


In [ ]:
bundle = next(Path("/kaggle/input").rglob("arc3-jinbo-bundle.json")).parent
sys.path.insert(0, str(bundle))

model_dir = None
for cfg in Path("/kaggle/input").rglob("config.json"):
    if list(cfg.parent.glob("*.safetensors")):
        model_dir = cfg.parent
        break
assert model_dir, "找不到模型目录(要挂 Kaggle Model)"

# dsh 离线 bundle(构建 kernel arc3-dsh-build 的 output 挂进来): node + dsh 构建产物
dsh_tgz = next(Path("/kaggle/input").rglob("dsh-bundle.tgz"))
DSH_ROOT = Path("/kaggle/tmp/dsh"); DSH_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.check_call(f"tar xzf {dsh_tgz} -C {DSH_ROOT}", shell=True)
NODE = DSH_ROOT / "node/bin/node"
DSH_BIN = DSH_ROOT / "dsh-src/apps/cli/lib/bin.js"
subprocess.check_call(f"{NODE} {DSH_BIN} --version", shell=True)
os.environ["A3_DSH_NODE"] = str(NODE)
os.environ["A3_DSH_BIN"] = str(DSH_BIN)
os.environ["A3_DSH_PATCH"] = str(bundle / "kaggle_agent/dsh/vllm.patch.yml")
os.environ["A3_DSH_TASK"] = str(bundle / "kaggle_agent/dsh/TASK_FULL.md")
os.environ["A3_DSH_HOME"] = "/kaggle/tmp/dsh-home"
os.environ["A3_AGENT"] = "dsh"
os.environ.setdefault("A3_CONCURRENCY", "16")
print("bundle:", bundle, "| model:", model_dir, "| dsh OK | concurrency", os.environ["A3_CONCURRENCY"])


## 3. 起 vLLM(工具调用 + 关思考模板; 权重加载约10-20分钟)

In [ ]:
from kaggle_agent.serve_vllm import start_vllm
# 关思考断根: 模型看到游戏画面后思考爆长把整个回复配额烧光(账本 finish=max-tokens 正文空)。
# vLLM 0.19 不认 --chat-template-kwargs, 但认 --chat-template 文件 —— 拷模板改两处判断
# (v20 打印出的 Qwen3.8 真实写法): 上面那处控制思考指令, 下面那处控制空 think 段
extra_vllm = []
tc = json.loads((model_dir / "tokenizer_config.json").read_text())
tpl = tc.get("chat_template") or ""
hits = 0
for a, b in (("enable_thinking is undefined or enable_thinking is true", "false"),
             ("enable_thinking is defined and enable_thinking is false", "true")):
    if a in tpl:
        tpl = tpl.replace(a, b); hits += 1
print("模板关思考替换命中:", hits)
assert hits == 2, "模板写法变了, 关思考没生效; 别带思考上赛场(会空转), 先看模板"
tf = WORKING / "chat_template_nothink.jinja"
tf.write_text(tpl)
extra_vllm = ["--chat-template", str(tf)]

# 上下文窗口: v1 线上测试 dsh 几轮后输入 28.7k + 输出 4000 > 32768 → CONTEXT_WINDOW_EXCEEDED 退出;
# dsh 自带的压缩请求(8192 输出)也同样撞墙。KV cache 实测能装 22 路 32k, 开 128k×4 路够用。
# 模型原生上限本地查不到, 从大到小试, 起成功的那档写进 dsh patch 的 contextWindow。
# Qwen3.8 工具调用是 XML 参数风格 -> qwen3_coder 解析器; 并附 qwen3 reasoning parser
proc, CTX = None, None
for ctx in (131072, 65536):
    try:
        proc = start_vllm(str(model_dir), port=8000, max_model_len=ctx, tool_calling=True,
                          tool_parser="qwen3_coder", extra_args=extra_vllm,
                          log_path=str(WORKING / f"vllm_{ctx}.log"), timeout_s=900)
        CTX = ctx
        print("vLLM up, max_model_len =", ctx)
        break
    except Exception as e:
        print(f"max_model_len={ctx} 起失败: {e!r}"[:400])
assert proc and CTX, "vLLM 两档窗口都起不来"
patch_src = (bundle / "kaggle_agent/dsh/vllm.patch.yml").read_text()
assert "contextWindow: 32768" in patch_src
patch_dst = WORKING / "vllm.patch.yml"
patch_dst.write_text(patch_src.replace("contextWindow: 32768", f"contextWindow: {CTX}"))
os.environ["A3_DSH_PATCH"] = str(patch_dst)

# 冒烟: 关思考真生效 = 简单问题秒答且 reasoning 为空; 原始响应打全, v1 出现过 content=None
from urllib.request import Request
req = Request("http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps({"model": "local", "max_tokens": 200,
        "messages": [{"role": "user", "content": "9+13*7等于几? 只回答数字"}]}).encode(),
    headers={"Content-Type": "application/json"})
t0 = time.time(); raw = json.loads(urlopen(req, timeout=300).read()); dt = time.time() - t0
m = raw["choices"][0]["message"]
print(f"冒烟 {dt:.1f}s finish={raw['choices'][0].get('finish_reason')} usage={raw.get('usage')}")
print("  message:", json.dumps(m, ensure_ascii=False)[:600])


## 4. 真提交: 网关(变量必须硬编码) / 离线: 公开环境文件


In [ ]:
def wait_gateway(base_url, timeout_s=600.0):
    deadline, last = time.monotonic() + timeout_s, ""
    probe = base_url.rstrip("/") + "/api/games"
    while time.monotonic() < deadline:
        try:
            with urlopen(probe, timeout=10) as r:
                if r.status < 500:
                    return
        except Exception as e:
            last = repr(e)
        time.sleep(5)
    raise RuntimeError(f"gateway not ready: {last}")

env_dir = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    os.environ.setdefault("SCHEME", "http")
    os.environ.setdefault("HOST", "gateway")
    os.environ.setdefault("PORT", "8001")
    os.environ.setdefault("OPERATION_MODE", "competition")
    os.environ.setdefault("ENVIRONMENTS_DIR", "")
    wait_gateway(os.environ["ARC_BASE_URL"])
    print("gateway ready")
else:
    cands = [p for p in Path("/kaggle/input").rglob("environment_files") if p.is_dir()] if Path("/kaggle/input").is_dir() else []
    cands += [bundle / "environment_files_sample", Path("environment_files")]
    env_dir = next((str(p) for p in cands if Path(p).is_dir()), None)
    assert env_dir, "找不到离线环境文件目录"
    print("offline env_dir:", env_dir)


## 5. 跑游戏

真提交: 总墙钟 8h, 按"组"均分(一组 = A3_CONCURRENCY 局同时跑)。离线验证: 3 局并发小预算把栈跑通。

In [ ]:
from kaggle_agent.run_submission import main

conc = int(os.environ["A3_CONCURRENCY"])
if TRUE_SUBMISSION:
    # 110 局 / 4 并发 ≈ 28 组, 8h 均分每组约 17 分钟; seconds_per_game 给上限, 实际按剩余时间均分
    budget = dict(seconds_per_game=1200.0, max_actions=200,
                  total_seconds=8 * 3600 - (time.time() - NOTEBOOK_T0) - 600)
    games = None
else:
    # 全部 25 公开局: 16+9 两组, 每组 480s, 顺便量 vLLM 16 路批处理的实际吞吐
    budget = dict(seconds_per_game=float(os.environ.get("A3_SECONDS_PER_GAME", 480)),
                  max_actions=int(os.environ.get("A3_MAX_ACTIONS", 200)),
                  total_seconds=float(os.environ.get("A3_TOTAL_SECONDS", 1100)))
    games = None

summary = main(env_dir=env_dir or "environment_files", games=games,
               out_dir=str(WORKING), concurrency=conc, **budget)


## 6. submission.parquet 门禁 + 结果


In [ ]:
if not TRUE_SUBMISSION:
    import pandas as pd
    pd.DataFrame([["1_0", "1", True, 1]],
                 columns=["row_id", "game_id", "end_of_game", "score"],
                 ).to_parquet(WORKING / "submission.parquet", index=False)
    print("submission.parquet written")
for g in summary["games"]:
    print(f"{g['game_id']:>16} levels {g['levels_completed']}/{g['win_levels']}"
          f" steps={g['steps']} {str(g['state'])[:30]} {g['seconds']}s")

# 诊断: 每局 dsh 会话账本尾部(模型每轮说了什么/调了什么/烧了多少 token)
try:
    import glob as _g, zstandard as _zstd
    for f in sorted(_g.glob("/kaggle/tmp/dsh-home/*/sessions/**/session*.jsonl.zstd", recursive=True)):
        short = f.split("/kaggle/tmp/dsh-home/")[1].split("/")[0]
        with open(f, "rb") as fh:
            data = _zstd.ZstdDecompressor().stream_reader(fh).read()
        (WORKING / f"session_{short}_tail.txt").write_text(data[-12000:].decode("utf-8", "replace"))
        print(f"[{short}] 账本尾已存, 总长 {len(data)}")
except Exception as e:
    print("账本解压失败:", repr(e))
